Add music generation notebook

In [4]:
pip install numpy sounddevice

Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import sounddevice as sd
import time

SR = 44100


# --------------------------------
# DEFINE THE SCALES
# --------------------------------

scales = {

    # D Dorian
    # D  E  F  G  A  B  C
    "D dorian": [
        293.66,
        329.63,
        349.23,
        392.00,
        440.00,
        493.88,
        523.25
    ],

    # C Lydian
    # C  D  E  F# G  A  B
    "C lydian": [
        261.63,
        293.66,
        329.63,
        369.99,
        392.00,
        440.00,
        493.88
    ],

    # C Major Pentatonic
    # C  D  E  G  A
    "C pentatonic": [
        261.63,
        293.66,
        329.63,
        392.00,
        440.00
    ]
}


# --------------------------------
# CHOOSE YOUR SCALE
# --------------------------------

scale = scales["D dorian"]


# --------------------------------
# CONVERT SCALE DEGREE TO FREQUENCY
# --------------------------------

def scale_note(degree):

    # Convert from human-friendly degree
    # (1, 2, 3...) to Python index
    index = degree - 1

    # Number of notes in this scale
    notes_per_octave = len(scale)

    # Which octave?
    octave = index // notes_per_octave

    # Which note within the octave?
    note = index % notes_per_octave

    return scale[note] * (2 ** octave)


# --------------------------------
# DEFINE THE NOTE FEEL
# --------------------------------

def dreamy_note(freq, duration=2.0, volume=0.25):

    t = np.linspace(
        0,
        duration,
        int(SR * duration),
        endpoint=False
    )

    # Gentle pitch movement
    vibrato = 0.6 * np.sin(2 * np.pi * 4.5 * t)

    phase = 2 * np.pi * freq * t + vibrato

    # Soft, bell-like timbre
    sound = (
        1.00 * np.sin(phase) +
        0.35 * np.sin(2 * phase) +
        0.18 * np.sin(3 * phase) +
        0.08 * np.sin(4 * phase) +
        0.04 * np.sin(5 * phase)
    )

    # Soft attack
    attack = 1 - np.exp(-18 * t)

    # Natural decay
    decay = np.exp(-1.4 * t)

    sound *= attack * decay

    # Organic noise
    sound += (
        np.random.normal(0, 0.002, len(sound))
        * attack
        * decay
    )

    sound *= volume

    # Spacious reverb
    delays = [
        (0.13, 0.18),
        (0.27, 0.12),
        (0.43, 0.08),
        (0.67, 0.05)
    ]

    result = np.zeros(
        len(sound) + int(0.67 * SR)
    )

    result[:len(sound)] += sound

    for delay_time, amount in delays:

        delay = int(delay_time * SR)

        result[
            delay:delay + len(sound)
        ] += sound * amount

    return result


# --------------------------------
# MELODY
# --------------------------------

melody = [
    1,
    3,
    5,
    7,
    5,
    3,
    2,
    6,
    1
]


# --------------------------------
# PLAY SETTINGS
# --------------------------------

duration = 2.0
volume = 0.20


# --------------------------------
# PLAY THE MELODY
# --------------------------------

for degree in melody:

    freq = scale_note(degree)

    sound = dreamy_note(
        freq,
        duration,
        volume
    )

    sd.play(sound, SR)

    # Notes overlap slightly
    time.sleep(duration * 0.55)

sd.stop()